# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect data, check for leakage, validate in time order, compare against baselines, report honestly.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, '| nulls:', int(df.isna().sum().sum()), '| unique assets:', df.asset_tag.nunique(),
      '| range:', df.inspected_at.min(), '->', df.inspected_at.max())
print('base rate:', round(df.failed_90d.mean(), 4))
df.describe().T

## 1. Leakage check
`parts_replaced` is the number of parts replaced on the work order *that closed this inspection record*, so it is recorded **after** the inspection and is not available when we would make a prediction. Let's see what it does.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean', 'size'])

Every record with 2+ parts replaced failed, and none with 0 did. That is a near-perfect proxy for the outcome (repairs that follow a failure), not a predictor. **It is excluded from the honest model**; the inflated score is shown below only to illustrate the trap.

Each asset appears once, so there is no asset overlap between train and test. Failure rate does shift strongly over time, so we validate with a time split.

In [ ]:
q = df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean', 'size'])
ax = q['mean'].plot(kind='bar', color='#3b6ea5', figsize=(7, 3), title='Failure rate by inspection quarter'); ax.set_ylabel('rate'); plt.show()
q

The rate swings between roughly 3% and 27% across quarters. The latest quarter may also be partly **right-censored**: inspections in the last 90 days of the window may not have had a full follow-up, depending on when the data was extracted. Worth confirming with the data owner.

In [ ]:
feat_num = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
d = df.copy()
d['bearing_minus_ambient'] = d.bearing_temp_c - d.ambient_temp_c
d = pd.concat([d, pd.get_dummies(d.site, prefix='site', dtype=float)], axis=1)
site_cols = [c for c in d if c.startswith('site_')]
F = feat_num + ['bearing_minus_ambient'] + site_cols
display(d.groupby('failed_90d')[feat_num].mean().T)
d[feat_num].corrwith(d.failed_90d).rename('corr with target').to_frame()

## 2. Time-ordered evaluation
Train on the earliest 70% of inspections, test on the latest 30%, then a rolling-origin check for stability.

In [ ]:
cut = int(len(d) * .7); tr, te = d.iloc[:cut], d.iloc[cut:]
print('train rate %.3f | test rate %.3f | test starts %s' % (tr.failed_90d.mean(), te.failed_90d.mean(), te.inspected_at.min().date()))
def models():
    return {'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
            'hgb': HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200, random_state=0)}
def evaluate(feats, tr, te):
    out = {}
    for n, m in models().items():
        m.fit(tr[feats], tr.failed_90d); p = m.predict_proba(te[feats])[:, 1]
        out[n] = (roc_auc_score(te.failed_90d, p), average_precision_score(te.failed_90d, p))
    return pd.DataFrame(out, index=['ROC AUC', 'PR AUC']).T
print('A random model scores ROC AUC 0.5 and PR AUC ~', round(te.failed_90d.mean(), 3))
evaluate(F, tr, te)

In [ ]:
print('WITH leaky parts_replaced (do not trust):')
evaluate(F + ['parts_replaced'], tr, te)

In [ ]:
# rolling origin: expanding train window, 4 successive test blocks
edges = np.linspace(0.5, 1, 5); rows = []
for a, b in zip(edges[:-1], edges[1:]):
    t, e = d.iloc[:int(len(d) * a)], d.iloc[int(len(d) * a):int(len(d) * b)]
    r = evaluate(F, t, e); r['block'] = f'{a:.2f}-{b:.2f}'; r['base_rate'] = e.failed_90d.mean(); rows.append(r.reset_index())
pd.concat(rows).pivot(index='block', columns='index', values='ROC AUC')

## 3. Which signals matter?

In [ ]:
m = models()['hgb'].fit(tr[F], tr.failed_90d)
pi = permutation_importance(m, te[F], te.failed_90d, scoring='roc_auc', n_repeats=20, random_state=0)
imp = pd.Series(pi.importances_mean, F).sort_values()
imp.plot(kind='barh', figsize=(6, 4), color='#3b6ea5', title='Permutation importance (drop in test ROC AUC)'); plt.show()
imp.sort_values(ascending=False).head(8)

In [ ]:
# failure rate by predicted-risk decile on the test set
p = m.predict_proba(te[F])[:, 1]
t = te.assign(p=p, dec=pd.qcut(p, 10, labels=False, duplicates='drop'))
t.groupby('dec').agg(pred=('p', 'mean'), actual=('failed_90d', 'mean'), n=('p', 'size'))

## 4. Conclusions (from the run on this data)
- **Yes, modestly.** Using only information available at inspection time, a gradient-boosted model reaches ROC AUC ~0.75 and PR AUC ~0.15 on the latest 30% of inspections (random is 0.5 and ~0.056). Logistic regression is weaker (~0.68), so there is some non-linear structure. Rolling-origin ROC AUC for the boosted model ranges ~0.70 to 0.83.
- **`parts_replaced` is leakage.** With it, AUC is ~0.997, but it is only known after the work order closes, so that number is not achievable in practice.
- **Drivers:** `load_pct` dominates permutation importance, followed by ambient and bearing temperature. Vibration correlates with failure marginally but adds little once the others are in the model. Site carries no signal.
- **Calibration is off under drift.** Failure rate fell from 14.5% (train) to 5.6% (test), so predicted probabilities run roughly 2x too high in the top deciles. Ranking is usable; raw probabilities need recalibration on recent data.
- **Caveats:** the strong quarter-to-quarter swings in failure rate are unexplained by the features, and the last quarter may be right-censored. Confirm the extraction date and what drove the Q2-Q3 2024 spike before relying on this. Each asset appears once, so we cannot use per-asset history (trends in vibration, time since last failure), which would likely help most.
